# Multimodal Deception Detection: Dual-Stream CrossAttention Pipeline
**Features:** MediaPipe Blendshapes (104) + Librosa Audio (34) + ResNet-18 (512) = **650 dims**

**Models:** PyTorch Dual-Stream CrossAttention · SVM · HistGradientBoosting · Random Forest · Hard-Voting Ensemble

**Additions:** Ablation Study · Publication Figures (300 DPI) · SHAP Global + Per-Clip XAI

```
My Drive/Deception_Capstone/
  ├── labels.csv       (video_id, label [0=Truth, 1=Lie])
  ├── videos/          (.mp4 files)
  └── features_resnet.csv   (auto-generated cache)
```

In [ ]:
# [CELL 1] Install Dependencies & Mount Drive
!pip install -q mediapipe librosa shap tqdm pandas numpy scikit-learn matplotlib seaborn
!pip install -q opencv-python-headless

# Download MediaPipe model ONCE here (not inside the function)
import os
if not os.path.exists('face_landmarker.task'):
    !wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
    print("face_landmarker.task downloaded.")
else:
    print("face_landmarker.task already present.")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# [CELL 2] Imports & Directory Configuration
import os, copy, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
import mediapipe as mp
import librosa
import cv2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              roc_curve, confusion_matrix,
                              precision_score, recall_score)
from scipy.stats import mode
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.notebook import tqdm

BASE_DIR     = '/content/drive/MyDrive/Deception_Capstone'
VIDEO_DIR    = os.path.join(BASE_DIR, 'videos')
LABELS_CSV   = os.path.join(BASE_DIR, 'labels.csv')
FEATURES_CSV = os.path.join(BASE_DIR, 'features_resnet.csv')
FIG_DIR      = os.path.join(BASE_DIR, 'paper_figures')
os.makedirs(FIG_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Figures will be saved to: {FIG_DIR}')

In [ ]:
# [CELL 3] Reproducibility — lock all seeds
import random

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print('Seeds locked to 42.')

## Phase 1: 650-Dimensional Feature Extraction
- **Audio (34 dims):** Librosa MFCC, ZCR, RMS, spectral centroid
- **MediaPipe (104 dims):** 52 blendshape mean + std per clip
- **ResNet-18 (512 dims):** Mean-pooled deep spatial embeddings (frozen, 20 sampled frames)

In [ ]:
# [CELL 4] Feature Extraction Pipeline
# face_landmarker.task is downloaded in Cell 1 — safe to reference here

def extract_single_video(video_path):
    """Extract 650-dim feature vector: MediaPipe(104)+Audio(34)+ResNet(512)."""

    # 1. AUDIO (34 dims)
    try:
        y, sr = librosa.load(video_path, sr=16000)
        mfcc      = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        rms       = librosa.feature.rms(y=y)
        zcr       = librosa.feature.zero_crossing_rate(y=y)
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        audio_features = (
            [float(np.mean(mfcc)), float(np.std(mfcc)),
             float(np.mean(rms)),  float(np.std(rms)),
             float(np.mean(zcr)),  float(np.std(zcr)),
             float(np.mean(spec_cent)), float(np.std(spec_cent))]
            + [float(x) for x in np.mean(mfcc, axis=1)]
            + [float(x) for x in np.std(mfcc,  axis=1)]
        )
    except Exception:
        audio_features = [0.0] * 34

    # 2. MEDIAPIPE BLENDSHAPES (104 dims)
    mediapipe_features = [0.0] * 104
    try:
        BaseOptions           = mp.tasks.BaseOptions
        FaceLandmarker        = mp.tasks.vision.FaceLandmarker
        FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
        options = FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path='face_landmarker.task'),
            output_face_blendshapes=True)
        cap = cv2.VideoCapture(video_path)
        blendshape_frames = []
        with FaceLandmarker.create_from_options(options) as landmarker:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                res       = landmarker.detect(mp_image)
                if res.face_blendshapes:
                    blendshape_frames.append(
                        [c.score for c in res.face_blendshapes[0]])
        cap.release()
        if blendshape_frames:
            arr = np.array(blendshape_frames)
            mediapipe_features = list(np.mean(arr, axis=0)) + list(np.std(arr, axis=0))
    except Exception:
        pass

    # 3. RESNET-18 SPATIAL EMBEDDINGS (512 dims — mean pooling over 20 frames)
    resnet_features = [0.0] * 512
    try:
        weights  = ResNet18_Weights.IMAGENET1K_V1
        rn_model = resnet18(weights=weights)
        rn_model.fc = nn.Identity()
        rn_model = rn_model.to(DEVICE).eval()
        preprocess = weights.transforms()

        cap2 = cv2.VideoCapture(video_path)
        n_frames   = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
        sample_idx = set(np.linspace(0, max(0, n_frames - 1), 20, dtype=int))
        tensors = []
        for i in range(n_frames):
            ret, frm = cap2.read()
            if not ret: break
            if i in sample_idx:
                t = torch.from_numpy(
                    cv2.cvtColor(frm, cv2.COLOR_BGR2RGB)
                ).permute(2,0,1).float() / 255.0
                tensors.append(preprocess(t))
        cap2.release()
        if tensors:
            batch = torch.stack(tensors).to(DEVICE)
            with torch.no_grad():
                resnet_features = rn_model(batch).mean(dim=0).cpu().numpy().tolist()
    except Exception:
        pass

    # Layout: [MediaPipe(104)] + [Audio(34)] + [ResNet(512)] = 650
    return mediapipe_features + audio_features + resnet_features


# Column name helpers — used throughout (SHAP labels, ablation slicing)
MP_COLS  = [f'mp_mean_{i}' for i in range(52)] + [f'mp_std_{i}'  for i in range(52)]
AUD_COLS = [f'audio_{i}'   for i in range(34)]
RES_COLS = [f'resnet_{i}'  for i in range(512)]
ALL_FEAT_COLS = MP_COLS + AUD_COLS + RES_COLS   # 650 total


def decode_feature(name):
    """Map raw column names to human-readable SHAP labels."""
    if name.startswith('mp_mean_'):
        return f'Face blendshape mean #{name.split("_")[-1]}'
    if name.startswith('mp_std_'):
        return f'Face blendshape std #{name.split("_")[-1]}'
    if name.startswith('audio_'):
        i = int(name.split('_')[-1])
        labels = ['MFCC global mean','MFCC global std','RMS mean','RMS std',
                  'ZCR mean','ZCR std','Spec-cent mean','Spec-cent std']
        if i < 8:  return labels[i]
        if i < 21: return f'MFCC-{i-8} coeff mean'
        return     f'MFCC-{i-21} coeff std'
    if name.startswith('resnet_'):
        return f'ResNet-18 dim #{name.split("_")[-1]}'
    return name


# Auto-extract if cache missing
if not os.path.exists(FEATURES_CSV) and os.path.exists(LABELS_CSV):
    print('Cache missing — extracting features (may take ~60 min on CPU)...')
    lbl_df   = pd.read_csv(LABELS_CSV)
    all_rows = []
    for _, row in tqdm(lbl_df.iterrows(), total=len(lbl_df), desc='Clips'):
        vid_id = str(row['video_id']).strip()
        fname  = vid_id if vid_id.endswith('.mp4') else f'{vid_id}.mp4'
        fpath  = os.path.join(VIDEO_DIR, fname)
        if os.path.exists(fpath):
            feats = extract_single_video(fpath)
            all_rows.append(feats + [int(row['label']), vid_id])

    out_df = pd.DataFrame(all_rows, columns=ALL_FEAT_COLS + ['label', 'video_id'])
    out_df.to_csv(FEATURES_CSV, index=False)
    print(f'Saved {len(out_df)} clips × 650 features → {FEATURES_CSV}')
else:
    print(f'Cache found: {FEATURES_CSV}')

## Phase 2: PyTorch Dual-Stream CrossAttention Architecture
- **Video encoder:** MediaPipe(104) + ResNet(512) → 616d → Linear → BN → GELU → ResidualBlock → 512d
- **Audio encoder:** Audio(34) → Linear → BN → GELU → ResidualBlock → 512d
- **CrossAttention:** V→A and A→V MultiheadAttention (4 heads) with residual addition
- **Classifier:** 1024 → 256 → 2

In [ ]:
# [CELL 5] PyTorch Model Definitions

class DeceptionDataset(Dataset):
    """Feature layout: [MP 0:104] [Audio 104:138] [ResNet 138:650]"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self):  return len(self.y)
    def __getitem__(self, idx):
        v = torch.cat([self.X[idx, :104], self.X[idx, 138:]])  # 616d video
        a = self.X[idx, 104:138]                                # 34d  audio
        return v, a, self.y[idx]


class ResidualBlock1D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.Dropout(0.4))
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(self.net(x) + x)


class ResNetDeceptionDetector(nn.Module):
    """Dual-Stream CrossAttention model (proposed)."""
    def __init__(self):
        super().__init__()
        dim = 512
        self.video_enc = nn.Sequential(
            nn.Linear(104+512, dim), nn.BatchNorm1d(dim), nn.GELU(), ResidualBlock1D(dim))
        self.audio_enc = nn.Sequential(
            nn.Linear(34, dim), nn.BatchNorm1d(dim), nn.GELU(), ResidualBlock1D(dim))
        self.cross_v2a = nn.MultiheadAttention(dim, num_heads=4, batch_first=True)
        self.cross_a2v = nn.MultiheadAttention(dim, num_heads=4, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(dim*2, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(256, 2))
    def forward(self, v, a):
        v = self.video_enc(v).unsqueeze(1)
        a = self.audio_enc(a).unsqueeze(1)
        v_attn, _ = self.cross_v2a(v, a, a)
        a_attn, _ = self.cross_a2v(a, v, v)
        out = torch.cat([v.squeeze(1) + v_attn.squeeze(1),
                         a.squeeze(1) + a_attn.squeeze(1)], dim=-1)
        return self.classifier(out)


def train_pytorch(X_train, y_train, X_val, y_val, epochs=100):
    """Train model, return best-checkpoint model + epoch history for plots."""
    model     = ResNetDeceptionDetector().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    tr_loader  = DataLoader(DeceptionDataset(X_train, y_train), batch_size=32, shuffle=True)
    val_loader = DataLoader(DeceptionDataset(X_val,   y_val),   batch_size=32)

    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    best_acc, best_state = 0.0, None

    for epoch in tqdm(range(epochs), desc='PyTorch Training'):
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        for v, a, yb in tr_loader:
            v, a, yb = v.to(DEVICE), a.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out  = model(v, a)
            loss = criterion(out, yb)
            loss.backward(); optimizer.step()
            t_loss    += loss.item() * len(yb)
            t_correct += (out.argmax(1) == yb).sum().item()
            t_total   += len(yb)

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for v, a, yb in val_loader:
                v, a, yb = v.to(DEVICE), a.to(DEVICE), yb.to(DEVICE)
                out  = model(v, a)
                loss = criterion(out, yb)
                v_loss    += loss.item() * len(yb)
                v_correct += (out.argmax(1) == yb).sum().item()
                v_total   += len(yb)

        history['train_loss'].append(t_loss / t_total)
        history['val_loss'].append(v_loss   / v_total)
        history['train_acc'].append(t_correct / t_total)
        history['val_acc'].append(v_correct   / v_total)

        if history['val_acc'][-1] > best_acc:
            best_acc   = history['val_acc'][-1]
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print(f'Best val acc: {best_acc:.4f}')
    return model, history


def get_pt_probs(model, X, y_dummy=None):
    """Return class-1 probabilities from a trained PyTorch model."""
    if y_dummy is None: y_dummy = np.zeros(len(X), dtype=int)
    loader = DataLoader(DeceptionDataset(X, y_dummy), batch_size=64)
    probs  = []
    model.eval()
    with torch.no_grad():
        for v, a, _ in loader:
            p = torch.softmax(model(v.to(DEVICE), a.to(DEVICE)), dim=1)[:,1]
            probs.extend(p.cpu().numpy())
    return np.array(probs)

## Phase 3: Ensemble Training & Full Evaluation
Trains all 4 models on the same stratified 70/15/15 split.
Scaler fitted on train only — never re-fitted on val or test.

In [ ]:
# [CELL 6] Ensemble Training & Evaluation
df = pd.read_csv(FEATURES_CSV)
print(f'Dataset: {df.shape[0]} clips, class balance: {dict(df["label"].value_counts())}')

features = [c for c in df.columns if c not in ['video_id','label']]
X = df[features].values.astype(np.float32)
y = df['label'].values.astype(int)

# Stratified split: 70% train | 15% val | 15% test
X_trainval, X_test,  y_trainval, y_test  = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)
X_train,    X_val,   y_train,    y_val   = train_test_split(
    X_trainval, y_trainval,
    test_size=0.15/(1-0.15), stratify=y_trainval, random_state=42)

scaler     = StandardScaler()
X_train_s  = scaler.fit_transform(X_train)
X_val_s    = scaler.transform(X_val)
X_test_s   = scaler.transform(X_test)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

# ── sklearn models ─────────────────────────────────────────────────────────────
print('\nTraining sklearn classifiers...')
hgb = HistGradientBoostingClassifier(
    max_iter=500, learning_rate=0.05, early_stopping=True, random_state=42
).fit(X_train_s, y_train)

svm = SVC(kernel='rbf', C=5.0, gamma='scale', probability=True, random_state=42
).fit(X_train_s, y_train)

rf = RandomForestClassifier(
    n_estimators=500, max_depth=15, random_state=42, n_jobs=-1
).fit(X_train_s, y_train)

# ── PyTorch CrossAttention ─────────────────────────────────────────────────────
print('\nTraining PyTorch Dual-Stream CrossAttention...')
pt_model, pt_history = train_pytorch(X_train_s, y_train, X_val_s, y_val)

# ── Test-set probabilities & predictions ──────────────────────────────────────
hgb_probs = hgb.predict_proba(X_test_s)[:,1]
svm_probs = svm.predict_proba(X_test_s)[:,1]
rf_probs  = rf.predict_proba(X_test_s)[:,1]
pt_probs  = get_pt_probs(pt_model, X_test_s, y_test)

hgb_preds = (hgb_probs >= 0.5).astype(int)
svm_preds = (svm_probs >= 0.5).astype(int)
rf_preds  = (rf_probs  >= 0.5).astype(int)
pt_preds  = (pt_probs  >= 0.5).astype(int)

# Hard voting ensemble
all_votes      = np.column_stack([hgb_preds, svm_preds, rf_preds, pt_preds])
final_preds, _ = mode(all_votes, axis=1)
final_preds    = final_preds.ravel()
ensemble_probs = (hgb_probs + svm_probs + rf_probs + pt_probs) / 4.0

# ── Full metrics table (all models including RF) ───────────────────────────────
print('\n' + '='*62)
print(f'  {"Model":<32} {"Acc":>6}  {"F1":>6}  {"AUC":>6}')
print('-'*62)
rows = []
for name, preds, probs in [
    ('PyTorch CrossAttention (proposed)', pt_preds,    pt_probs),
    ('Support Vector Machine',            svm_preds,   svm_probs),
    ('HistGradientBoosting',              hgb_preds,   hgb_probs),
    ('Random Forest',                     rf_preds,    rf_probs),     # ← FIX: was missing
    ('Hard-Voting Ensemble',              final_preds, ensemble_probs),
]:
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    print(f'  {name:<32} {acc:6.4f}  {f1:6.4f}  {auc:6.4f}')
    rows.append({'Model':name, 'Accuracy':f'{acc*100:.2f}%',
                 'F1':f'{f1:.4f}', 'AUC':f'{auc:.4f}'})
print('='*62)

display(pd.DataFrame(rows))

### 5-Fold Cross-Validation (quick stability check)

In [ ]:
# [CELL 7] 5-Fold CV
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv5 = []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(X, y)):
    s = StandardScaler()
    Xtr = s.fit_transform(X[tr_idx]); Xvl = s.transform(X[vl_idx])
    sc  = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr, y[tr_idx]).score(Xvl, y[vl_idx])
    cv5.append(sc)
    print(f'  Fold {fold+1}: {sc:.4f}')
print(f'  5-Fold Mean: {np.mean(cv5):.4f} ± {np.std(cv5):.4f}')

### 10-Fold Cross-Validation (primary reliability estimate — SVM)

In [ ]:
# [CELL 8] 10-Fold CV
skf10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv10  = []

print('10-Fold Stratified CV (SVM):')
for fold, (tr_idx, vl_idx) in enumerate(skf10.split(X, y)):
    s   = StandardScaler()
    Xtr = s.fit_transform(X[tr_idx]); Xvl = s.transform(X[vl_idx])
    sc  = SVC(kernel='rbf', C=5.0, probability=False, random_state=42).fit(
            Xtr, y[tr_idx]).score(Xvl, y[vl_idx])
    cv10.append(sc)
    print(f'  Fold {fold+1:2d}: {sc:.4f}')

print(f'\n10-Fold Mean Accuracy : {np.mean(cv10):.4f}')
print(f'10-Fold Std Dev        : ±{np.std(cv10):.4f}')
print(f'95% CI (approx)        : ({np.mean(cv10)-2*np.std(cv10):.4f}, {np.mean(cv10)+2*np.std(cv10):.4f})')

## Phase 4: Ablation Study

Tests 6 feature configurations under **5-fold stratified CV** (scaler refit inside each fold):

| # | Configuration | Dims | Tests |
|---|---|---|---|
| 1 | Full (all streams) | 650 | Proposed system |
| 2 | MediaPipe only | 104 | Facial stream alone |
| 3 | Audio only | 34 | Audio stream alone |
| 4 | ResNet only | 512 | Visual deep stream alone |
| 5 | No MediaPipe | 546 | Audio + ResNet |
| 6 | No Audio | 616 | MediaPipe + ResNet |

In [ ]:
# [CELL 9] Ablation Study — 5-Fold CV across 6 feature configurations
# Feature index slices in 650-dim vector:
#   MediaPipe = 0:104
#   Audio     = 104:138
#   ResNet    = 138:650

ABLATION = {
    'Full (650d)':       list(range(650)),
    'MediaPipe only':    list(range(104)),
    'Audio only':        list(range(104, 138)),
    'ResNet only':       list(range(138, 650)),
    'No MediaPipe':      list(range(104, 650)),
    'No Audio':          list(range(104)) + list(range(138, 650)),
}

SKF_ABL = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ablation_results = {}

print(f'  {"Config":<20} {"HGB":>7}  {"SVM":>7}  {"RF":>7}')
print('-'*48)
for cfg_name, col_idx in ABLATION.items():
    X_cfg = X[:, col_idx]
    hgb_s, svm_s, rf_s = [], [], []

    for tr_idx, vl_idx in SKF_ABL.split(X_cfg, y):
        s = StandardScaler()
        Xtr = s.fit_transform(X_cfg[tr_idx])
        Xvl = s.transform(X_cfg[vl_idx])
        ytr, yvl = y[tr_idx], y[vl_idx]

        hgb_s.append(HistGradientBoostingClassifier(
            max_iter=300, random_state=42).fit(Xtr, ytr).score(Xvl, yvl))
        svm_s.append(SVC(kernel='rbf', C=5.0, random_state=42
            ).fit(Xtr, ytr).score(Xvl, yvl))
        rf_s.append(RandomForestClassifier(
            n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ytr).score(Xvl, yvl))

    ablation_results[cfg_name] = {
        'HGB': hgb_s, 'SVM': svm_s, 'RF': rf_s}
    print(f'  {cfg_name:<20} '
          f'{np.mean(hgb_s):7.4f}  {np.mean(svm_s):7.4f}  {np.mean(rf_s):7.4f}')

print('-'*48)
print('All values are 5-fold CV mean accuracy.')

## Phase 5: Publication-Quality Figures (300 DPI → saved to Drive)

| Figure | File | Section in paper |
|---|---|---|
| Fig A | `figA_ablation.pdf` | Results — Ablation Study |
| Fig B | `figB_roc_curves.pdf` | Results — ROC |
| Fig C | `figC_confusion_grid.pdf` | Results — Confusion matrices |
| Fig D | `figD_training_curves.pdf` | Results / Appendix |
| Fig E | `figE_feature_importance.pdf` | Results — SHAP/Importance |

In [ ]:
# [CELL 10] Publication Figures

PUB_STYLE = {
    'font.family':      'serif',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'grid.linestyle':   '--',
    'figure.dpi':       150,
}

def save_fig(fname):
    path = os.path.join(FIG_DIR, fname)
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved → {path}')


# ─────────────────────────────────────────────────────────────
# FIG A: Ablation bar chart
# ─────────────────────────────────────────────────────────────
with plt.rc_context(PUB_STYLE):
    cfg_names = list(ABLATION.keys())
    mdl_names = ['HGB', 'SVM', 'RF']
    colors    = ['#2166ac', '#d6604d', '#4dac26']
    means = {m: [np.mean(ablation_results[c][m]) for c in cfg_names] for m in mdl_names}
    stds  = {m: [np.std( ablation_results[c][m]) for c in cfg_names] for m in mdl_names}

    x, w = np.arange(len(cfg_names)), 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    for j, (m, col) in enumerate(zip(mdl_names, colors)):
        ax.bar(x + j*w, means[m], w, yerr=stds[m], label=m,
               color=col, alpha=0.85,
               error_kw=dict(elinewidth=1.2, capsize=4))
    ax.set_xticks(x + w); ax.set_xticklabels(cfg_names, rotation=22, ha='right')
    ax.set_ylabel('5-Fold CV Accuracy'); ax.set_ylim(0.45, 1.0)
    ax.axhline(0.5, color='gray', lw=0.8, ls=':', label='Chance (0.50)')
    ax.legend(framealpha=0.9)
    ax.set_title('Fig A — Ablation Study: 5-Fold CV Accuracy by Feature Configuration')
    fig.tight_layout(); save_fig('figA_ablation.pdf')


# ─────────────────────────────────────────────────────────────
# FIG B: ROC curves — all 5 models
# ─────────────────────────────────────────────────────────────
with plt.rc_context(PUB_STYLE):
    fig, ax = plt.subplots(figsize=(6, 5))
    roc_configs = [
        ('CrossAttention (proposed)', pt_probs,       '#d6604d', '-',  2.0),
        ('SVM',                       svm_probs,       '#2166ac', '-',  1.6),
        ('HGB',                       hgb_probs,       '#4dac26', '-',  1.6),
        ('Random Forest',             rf_probs,        '#762a83', '-',  1.6),
        ('Hard-Voting Ensemble',      ensemble_probs,  '#000000', '--', 1.6),
    ]
    for label, probs, col, ls, lw in roc_configs:
        fpr, tpr, _ = roc_curve(y_test, probs)
        auc = roc_auc_score(y_test, probs)
        ax.plot(fpr, tpr, color=col, lw=lw, ls=ls,
                label=f'{label}  (AUC = {auc:.3f})')
    ax.plot([0,1],[0,1], 'k:', lw=0.8)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('Fig B — ROC Curves (Test Set)')
    ax.legend(fontsize=9, loc='lower right')
    fig.tight_layout(); save_fig('figB_roc_curves.pdf')


# ─────────────────────────────────────────────────────────────
# FIG C: Confusion matrices — 2×3 grid (all 5 models + ensemble)
# ─────────────────────────────────────────────────────────────
with plt.rc_context(PUB_STYLE):
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    cm_pairs = [
        ('CrossAttention', pt_preds),
        ('SVM',            svm_preds),
        ('HGB',            hgb_preds),
        ('Random Forest',  rf_preds),
        ('Hard-Vote Ensemble', final_preds),
    ]
    for ax, (name, preds) in zip(axes.ravel(), cm_pairs):
        cm  = confusion_matrix(y_test, preds)
        acc = accuracy_score(y_test, preds)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Truth','Lie'], yticklabels=['Truth','Lie'],
                    cbar=False, linewidths=0.5)
        ax.set_title(f'{name}  (Acc={acc:.3f})', fontsize=10)
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

    # Hide the unused 6th subplot
    axes[1, 2].set_visible(False)
    fig.suptitle('Fig C — Confusion Matrices (Test Set)', fontsize=13, y=1.01)
    fig.tight_layout(); save_fig('figC_confusion_grid.pdf')


# ─────────────────────────────────────────────────────────────
# FIG D: Training curves (loss + accuracy)
# ─────────────────────────────────────────────────────────────
with plt.rc_context(PUB_STYLE):
    ep = range(1, len(pt_history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    ax = axes[0]
    ax.plot(ep, pt_history['train_loss'], '#d6604d', lw=1.8, label='Train loss')
    ax.plot(ep, pt_history['val_loss'],   '#d6604d', lw=1.8, ls='--', label='Val loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
    ax.set_title('Training & Validation Loss'); ax.legend()

    ax = axes[1]
    ax.plot(ep, [v*100 for v in pt_history['train_acc']], '#2166ac', lw=1.8, label='Train acc')
    ax.plot(ep, [v*100 for v in pt_history['val_acc']],   '#2166ac', lw=1.8, ls='--', label='Val acc')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.set_title('Training & Validation Accuracy'); ax.legend()

    fig.suptitle('Fig D — PyTorch CrossAttention Training Curves', fontsize=12)
    fig.tight_layout(); save_fig('figD_training_curves.pdf')


# ─────────────────────────────────────────────────────────────
# FIG E: RF feature importance by modality group
# ─────────────────────────────────────────────────────────────
with plt.rc_context(PUB_STYLE):
    imp = rf.feature_importances_
    groups = {
        'MediaPipe\n(blendshape mean)': (0,   52),
        'MediaPipe\n(blendshape std)':  (52,  104),
        'Audio\n(global stats)':        (104, 112),
        'Audio\n(MFCC per-coeff)':      (112, 138),
        'ResNet-18\n(spatial emb.)':    (138, 650),
    }
    grp_imp = {k: imp[v[0]:v[1]].sum() for k, v in groups.items()}
    total   = sum(grp_imp.values())
    grp_norm = {k: v/total for k, v in grp_imp.items()}

    grp_cols = ['#4dac26','#b8e186','#d6604d','#f4a582','#2166ac']
    fig, ax  = plt.subplots(figsize=(8, 4))
    bars     = ax.barh(list(grp_norm.keys()), list(grp_norm.values()),
                       color=grp_cols, edgecolor='white')
    for bar, val in zip(bars, grp_norm.values()):
        ax.text(val+0.003, bar.get_y()+bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    ax.set_xlabel('Normalized Feature Importance (RF impurity-based)')
    ax.set_title('Fig E — Feature Importance by Modality Group (Random Forest)')
    ax.set_xlim(0, max(grp_norm.values())*1.2)
    fig.tight_layout(); save_fig('figE_feature_importance.pdf')

print('\\nAll publication figures saved.')

## Phase 6: SHAP Explainability

- **Global beeswarm:** Top 20 features across the full test set
- **Modality-group SHAP bar:** Mean |SHAP| per stream
- **Per-clip waterfall:** One correctly-classified truthful + one deceptive example from test set

In [ ]:
# [CELL 11] SHAP Global Analysis

print('Computing SHAP values (RandomForest TreeExplainer on test set)...')
explainer_rf = shap.TreeExplainer(rf)
shap_values  = explainer_rf.shap_values(X_test_s)

# Handle both list (old sklearn) and 3D array (new sklearn) SHAP output
if isinstance(shap_values, list):
    sv_class1 = shap_values[1]          # class 1 = Deception
elif shap_values.ndim == 3:
    sv_class1 = shap_values[:, :, 1]
else:
    sv_class1 = shap_values

readable = [decode_feature(f) for f in features]

# ── Fig F: SHAP Beeswarm (global, top 20) ─────────────────────────────────────
with plt.rc_context({**PUB_STYLE, 'axes.grid': False}):
    plt.figure(figsize=(9, 7))
    shap.summary_plot(sv_class1, X_test_s, feature_names=readable,
                      max_display=20, show=False, plot_type='dot')
    plt.title('Fig F — SHAP Beeswarm: Top 20 Features (Deception class)', pad=12)
    plt.tight_layout()
    out = os.path.join(FIG_DIR, 'figF_shap_beeswarm.pdf')
    plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved → {out}')

# ── Fig G: SHAP by modality group (mean |SHAP|) ───────────────────────────────
group_shap = {
    'MediaPipe\n(blendshapes)': np.abs(sv_class1[:, :104]).mean(),
    'Audio\n(Librosa)':         np.abs(sv_class1[:, 104:138]).mean(),
    'ResNet-18\n(spatial)':     np.abs(sv_class1[:, 138:]).mean(),
}
with plt.rc_context(PUB_STYLE):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    grp_c = ['#4dac26', '#d6604d', '#2166ac']
    ax.barh(list(group_shap.keys()), list(group_shap.values()),
            color=grp_c, edgecolor='white')
    for i, (k, v) in enumerate(group_shap.items()):
        ax.text(v+0.0003, i, f'{v:.4f}', va='center', fontsize=9)
    ax.set_xlabel('Mean |SHAP value| — contribution to deception prediction')
    ax.set_title('Fig G — Mean SHAP Contribution by Modality Group')
    fig.tight_layout()
    out = os.path.join(FIG_DIR, 'figG_shap_modality_group.pdf')
    fig.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved → {out}')

In [ ]:
# [CELL 12] Per-Clip SHAP Waterfall — Truthful + Deceptive examples from test set

def shap_waterfall(clip_idx, title_str, fname):
    \"\"\"Generate & save a SHAP waterfall for a single test-set clip.\"\"\"
    sv_i = sv_class1[clip_idx]

    if isinstance(explainer_rf.expected_value, (list, np.ndarray)):
        base_val = float(explainer_rf.expected_value[1])
    else:
        base_val = float(explainer_rf.expected_value)

    exp = shap.Explanation(
        values       = sv_i,
        base_values  = base_val,
        data         = X_test_s[clip_idx],
        feature_names= readable)

    with plt.rc_context({**PUB_STYLE, 'axes.grid': False}):
        plt.figure(figsize=(10, 7))
        shap.plots.waterfall(exp, max_display=15, show=False)
        plt.title(f'{title_str}  (test clip #{clip_idx})', pad=12, fontsize=12)
        plt.tight_layout()
        out = os.path.join(FIG_DIR, fname)
        plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        print(f'Saved → {out}')


# Pick examples: correctly classified truth + correctly classified deception
truth_ok      = np.where((y_test == 0) & (pt_preds == 0))[0]
deception_ok  = np.where((y_test == 1) & (pt_preds == 1))[0]

if len(truth_ok) > 0:
    shap_waterfall(truth_ok[0],
                   'SHAP Waterfall — Correctly Classified: TRUTHFUL',
                   'figH_shap_waterfall_truth.pdf')
else:
    print('No correctly classified truthful example found in test set.')

if len(deception_ok) > 0:
    shap_waterfall(deception_ok[0],
                   'SHAP Waterfall — Correctly Classified: DECEPTIVE',
                   'figI_shap_waterfall_deceptive.pdf')
else:
    print('No correctly classified deceptive example found in test set.')

print('\\nAll SHAP figures saved.')

## Phase 7: Live Inference on Uploaded Clip

In [ ]:
# [CELL 13] Live Inference + SHAP Waterfall for Uploaded Clip
import google.colab.files as colab_files

uploaded = colab_files.upload()
if uploaded:
    fname_up = list(uploaded.keys())[0]
    print(f'Processing: {fname_up}')

    raw_feats  = extract_single_video(fname_up)
    feat_array = np.array(raw_feats, dtype=np.float32).reshape(1, -1)
    feat_scaled = scaler.transform(feat_array)

    hgb_p = hgb.predict_proba(feat_scaled)[0, 1]
    svm_p = svm.predict_proba(feat_scaled)[0, 1]
    rf_p  = rf.predict_proba(feat_scaled)[0, 1]
    pt_p  = get_pt_probs(pt_model, feat_scaled)[0]
    avg_p = (hgb_p + svm_p + rf_p + pt_p) / 4.0
    verdict = 'DECEPTIVE' if avg_p >= 0.5 else 'TRUTHFUL'

    print('\\n' + '='*45)
    print(f'  CrossAttention:  {pt_p:.2%}')
    print(f'  SVM:             {svm_p:.2%}')
    print(f'  HGB:             {hgb_p:.2%}')
    print(f'  Random Forest:   {rf_p:.2%}')
    print(f'  Ensemble avg:    {avg_p:.2%}')
    print(f'  VERDICT:         {verdict}')
    print('='*45)

    # Side-by-side: model agreement bars + SHAP waterfall
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    mdl_names_live = ['CrossAttention','SVM','HGB','RF','Ensemble']
    mdl_probs_live = [pt_p, svm_p, hgb_p, rf_p, avg_p]
    bar_cols = ['#d6604d' if p>=0.5 else '#2166ac' for p in mdl_probs_live]
    ax.barh(mdl_names_live, mdl_probs_live, color=bar_cols, edgecolor='white')
    ax.axvline(0.5, color='black', lw=1.2, ls='--')
    for i, p in enumerate(mdl_probs_live):
        ax.text(p+0.01, i, f'{p:.1%}', va='center', fontsize=10)
    ax.set_xlim(0, 1.18); ax.set_xlabel('P(Deception)')
    ax.set_title(f'Model Agreement — Verdict: {verdict}')
    red_p  = mpatches.Patch(color='#d6604d', label='Predicts Deception')
    blue_p = mpatches.Patch(color='#2166ac', label='Predicts Truth')
    ax.legend(handles=[red_p, blue_p], loc='lower right')

    # SHAP waterfall
    sv_live = explainer_rf.shap_values(feat_scaled)
    if isinstance(sv_live, list):
        sv_live_c1 = sv_live[1][0]
        base_live  = float(explainer_rf.expected_value[1])
    elif sv_live.ndim == 3:
        sv_live_c1 = sv_live[0, :, 1]
        base_live  = float(explainer_rf.expected_value[1])
    else:
        sv_live_c1 = sv_live[0]
        base_live  = (float(explainer_rf.expected_value[1])
                      if hasattr(explainer_rf.expected_value, '__len__')
                      else float(explainer_rf.expected_value))

    exp_live = shap.Explanation(
        values       = sv_live_c1,
        base_values  = base_live,
        data         = feat_scaled[0],
        feature_names= readable)

    plt.sca(axes[1])
    shap.plots.waterfall(exp_live, max_display=12, show=False)
    axes[1].set_title('SHAP: Top 12 features driving this prediction')

    fig.suptitle(f'Live Inference: {fname_up}  |  Verdict: {verdict}', fontsize=12)
    fig.tight_layout()
    out = os.path.join(FIG_DIR, f'inference_{os.path.splitext(fname_up)[0]}.pdf')
    fig.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved → {out}')
else:
    print('No file uploaded.')

## Phase 8: Hyperparameter Optimization

In [ ]:
# [CELL 14] RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV

final_scaler = StandardScaler()
X_opt = final_scaler.fit_transform(X_trainval)
y_opt = y_trainval

svm_space = {'C':[0.1,1,5,10], 'gamma':['scale','auto',0.01]}
svm_search = RandomizedSearchCV(SVC(probability=True), svm_space,
                                n_iter=5, cv=3, random_state=42)
svm_search.fit(X_opt, y_opt)
print(f'Best SVM: {svm_search.best_params_}  CV={svm_search.best_score_:.4f}')

rf_space = {'n_estimators':[100,300,500], 'max_depth':[10,20,None],
            'min_samples_split':[2,5]}
rf_search = RandomizedSearchCV(RandomForestClassifier(n_jobs=-1), rf_space,
                               n_iter=5, cv=3, random_state=42)
rf_search.fit(X_opt, y_opt)
print(f'Best RF:  {rf_search.best_params_}  CV={rf_search.best_score_:.4f}')

## Phase 9: Save All Model Artifacts

In [ ]:
# [CELL 15] Save Models to Drive
import joblib, json

SAVE_DIR = '/content/drive/MyDrive/Deception_Capstone/saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(hgb,    os.path.join(SAVE_DIR, 'hgb_model.joblib'))
joblib.dump(svm,    os.path.join(SAVE_DIR, 'svm_model.joblib'))
joblib.dump(rf,     os.path.join(SAVE_DIR, 'rf_model.joblib'))
joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler.joblib'))
print('Sklearn models saved.')

torch.save(pt_model.state_dict(), os.path.join(SAVE_DIR, 'pytorch_model.pth'))
print('PyTorch model saved.')

with open(os.path.join(SAVE_DIR, 'feature_names.json'), 'w') as f:
    json.dump(features, f)

meta = {
    'feature_dim': 650,
    'audio_dim':   34,    # indices 104:138
    'video_dim':   616,   # indices 0:104 + 138:650
    'num_classes': 2,
    'label_map':   {0:'Truth', 1:'Deception'}
}
with open(os.path.join(SAVE_DIR, 'model_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(f'All artifacts saved to: {SAVE_DIR}')
print('Paper figures saved to:', FIG_DIR)